# **Практика: использование Permutation importance для оценки важности признаков в линейной модели**

В ходе уроков модуля вы детально изучили permutation importance теоретически. В этом ноутбуке мы предлагаем использовать и проанализировать данный способ оценки важности признаков на практике!

В качестве набора данных будем использовать встроенный dataset из sklearn 'fetch_california_housing'. Он был составлен на основе данных переписи населения Калифорнии 1990 года. Каждая строка в наборе данных представляет отдельный жилой район и содержит информацию о средних характеристиках домов в этом районе.

**Основные характеристики:**

**Целевая переменная (target):**

- `MedHouseVal` — средняя стоимость домов в данном районе, выраженная в сотнях тысяч долларов.

**Признаки (features):**

- `MedInc` (Median Income): медианный доход в районе (в десятках тысяч долларов)

- `HouseAge` (Housing Median Age): медианный возраст домов в районе (в годах)
- `AveRooms` (Average Rooms per Dwelling): среднее количество комнат на одно жилье.
- `AveBedrms` (Average Bedrooms per Dwelling): среднее количество спален на одно жилье.
- `Population (Population):` численность населения в районе.
-  `AveOccup (Average Occupancy):` среднее количество жильцов на одно жилище.
-  `Latitude (Latitude):` географическая широта района.
- `Longitude (Longitude):` географическая долгота района.

Датасет не содержит пропусков и является отличным для нашей практической задачи – оценить важность признаков. Приятного прохождения заданий!

![jean-louis-paulin-lHwmE58fW4Y-unsplash.md.jpg](https://ucarecdn.com/246c0810-6036-4dc9-bed3-f7f7498e5790/)


# Предобработка данных

Предобработку оставим минимальной: признаки не меняем, а стандартизацию там, где она нужна, делает сама модель (`StandardScaler` внутри конвейера `make_pipeline`). Чтобы проверить, как метод ведет себя на заведомо бесполезных признаках, добавим два случайных:

- `rnd_bin` — бинарный признак, который в этой выборке всегда равен 0;
- `rnd_num` — гауссов шум, никак не связанный ни с целевой переменной, ни с другими признаками.

Разбиение на обучающую и тестовую выборки делаем с `random_state=42`, как в заданиях тренажера.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate, RepeatedKFold
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance

data = pd.read_csv('https://github.com/SadSabrina/explainable_AI_course/raw/refs/heads/main/data/fetch_california_housing.csv',
                   index_col=0)
data.head()


In [ ]:
X, y = data.drop('target', axis=1), data['target']

rng = np.random.RandomState(0)
bin_var = pd.Series(rng.randint(0, 1, X.shape[0]), name="rnd_bin")  # всегда 0
num_var = pd.Series(rng.randn(X.shape[0]), name="rnd_num")  # гауссов шум
X_with_rnd_feat = pd.concat((X, bin_var, num_var), axis=1)

# разделим на тренировочный и тестовый наборы
X_train, X_test, y_train, y_test = train_test_split(
    X_with_rnd_feat, y, random_state=42
)


# Анализ переменных

Чтобы не работать с признаками вслепую, извлечем базовую информацию о взаимосвязи:
- признаков друг с другом
- признаков с целевой переменной.


In [ ]:
#Создадим экземпляр, в который добавим целевую переменную
train_dataset = X_train.copy()
train_dataset.insert(0, "target", y_train)
g = sns.pairplot(
    train_dataset[
        ["target", "Latitude", "AveRooms", "AveBedrms", "MedInc", 'rnd_num']
    ],
    height=1.5
)

g.fig.suptitle('Взаимные распределения непрерывных признаков',
               y=1.05,
               fontsize=13
);


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(train_dataset.drop('rnd_bin', axis=1).corr(), annot=True)

ax.set_title('Взаимная корреляция непрерывных признаков');


**Место для ваших выводов:**
- какие пары признаков сильно скоррелированы (корреляция по модулю не ниже 0.5)?
- какой признак сильнее всего линейно связан с целевой переменной?


# Линейная модель

**Q1. Обучите линейную модель на California Housing (train/test split, `random_state=42`). Чему равен $R^2$ на тестовых данных? Ответ округлите до сотых.**

Линейная модель здесь — `LinearRegression` внутри конвейера со `StandardScaler`: стандартизация не меняет прогноз линейной регрессии, но делает коэффициенты сравнимыми между собой. Эта же модель `lin_model` используется во всех заданиях ниже.

Введите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/1) · [шаг на Степике](https://stepik.org/lesson/1329680/step/2)

In [ ]:
lin_model = make_pipeline(StandardScaler(), LinearRegression())
lin_model.fit(X_train, y_train)

r2_test = ...  # ваш код: R^2 модели на тестовых данных (метод .score)
print(f"R^2 на тестовых данных: {r2_test}")


Взглянем на коэффициенты модели. Признаки стандартизованы, поэтому коэффициенты можно сравнивать по модулю.


In [ ]:
coefs = pd.DataFrame(
    lin_model[1].coef_, columns=["Coefficients"], index=X_train.columns
)

coefs.plot(kind="barh", figsize=(9, 7))
plt.title("Коэффициенты линейной модели (стандартизованные признаки)")
plt.axvline(x=0, color=".5")
plt.subplots_adjust(left=0.3)
coefs


Насколько устойчивы коэффициенты? Обучим ту же модель на фолдах повторной кросс-валидации и посмотрим на разброс. Сравните разброс у скоррелированных пар `AveRooms`–`AveBedrms` и `Latitude`–`Longitude` с разбросом у остальных признаков.


In [ ]:
cv_model = cross_validate(
    lin_model,
    X_with_rnd_feat,
    y,
    cv=RepeatedKFold(n_splits=5, n_repeats=5, random_state=0),
    return_estimator=True,
    n_jobs=2,
)
cv_coefs = pd.DataFrame(
    [model[1].coef_ for model in cv_model["estimator"]],
    columns=X_with_rnd_feat.columns,
)
plt.figure(figsize=(9, 7))
sns.boxplot(data=cv_coefs, orient="h", color="cyan", saturation=0.5)
plt.axvline(x=0, color=".5")
plt.xlabel("Значение коэффициента")
plt.title("Коэффициенты и их изменчивость по фолдам")
plt.subplots_adjust(left=0.3)
cv_coefs.std().sort_values()


# Feature importance by permutation

Напомним алгоритм вычисления перестановочной важности признака $f_i$.

1. Обучим модель и вычислим ее ошибку на тестовых данных $e_{orig}$ (здесь — MSE).
2. Зафиксируем признак $f_i$ и число перестановок $K$.
3. Для каждого $k$ от 1 до $K$:
 - случайно переставим значения признака $f_i$ в тестовых данных;
 - вычислим прогноз **уже обученной** модели на данных с перестановкой (модель не переобучается);
 - вычислим ошибку $e_{perm,k}$.
4. Сравним среднюю ошибку после перестановки с исходной — отношением или разностью:

$$PI\_ratio_i = \frac{\frac{1}{K}\sum_{k=1}^{K} e_{perm,k}}{e_{orig}}, \qquad PI\_diff_i = \frac{1}{K}\sum_{k=1}^{K} e_{perm,k} - e_{orig}.$$

Функция `permuted_errors` ниже делает шаги 1–3 и возвращает $e_{orig}$ и массив из $K$ значений $e_{perm,k}$.


In [ ]:
def permuted_errors(model, X, y, feature, K=10, seed=42):
    rs = np.random.RandomState(seed)
    e_orig = mean_squared_error(y, model.predict(X))
    e_perm = []
    for k in range(K):
        X_perm = X.copy()
        X_perm[feature] = rs.permutation(X_perm[feature].values)  # перестановка одного признака
        e_perm.append(mean_squared_error(y, model.predict(X_perm)))
    return e_orig, np.array(e_perm)


**Q2. Посчитайте `PI_ratio` для признака `Latitude` вручную (через ошибку MSE, $K=10$ перестановок, `np.random.seed(42)`). Чему равно значение (округлите до сотых)?**

Считайте на тестовых данных для модели `lin_model`, с `K=10` и `seed=42`.

Введите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/2) · [шаг на Степике](https://stepik.org/lesson/1329680/step/3)

In [ ]:
e_orig, e_perm = permuted_errors(lin_model, X_test, y_test, 'Latitude', K=10, seed=42)

PI_ratio_lat = ...  # ваш код: PI_ratio по формуле выше
PI_diff_lat = ...  # ваш код: PI_diff по формуле выше
print(f"PI_ratio(Latitude) = {PI_ratio_lat}, PI_diff(Latitude) = {PI_diff_lat}")


Посчитаем обе величины для всех признаков сразу.

**Q3. Для `Population` и `rnd_num` получили `PI_ratio ≈ 1.00`, `PI_diff ≈ 0.00`. Выберите верные утверждения:**

Выберите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/3) · [шаг на Степике](https://stepik.org/lesson/1329680/step/4)

In [ ]:
def pi_table(model, X, y, K=10, seed=42):
    rows = {}
    for feature in X.columns:
        e_orig, e_perm = permuted_errors(model, X, y, feature, K=K, seed=seed)
        rows[feature] = {'PI_ratio': e_perm.mean() / e_orig, 'PI_diff': e_perm.mean() - e_orig}
    return pd.DataFrame(rows).T

pi_lin = pi_table(lin_model, X_test, y_test)
pi_lin.sort_values('PI_ratio', ascending=False).round(3)


# Цена перестановок

**Q4 (быстрый устный счет). Один вызов модели стоит 3.6 секунды, выборка — 144 объекта. Оцените выигрыш времени (в часах) при переходе от точного перебора пар ($n(n-1)$ вызовов) к аппроксимации «пополам» ($n$ вызовов). Ответ округлите до десятых.**

Код здесь не нужен; ячейка ниже — для проверки себя.

Введите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/4) · [шаг на Степике](https://stepik.org/lesson/1329680/step/5)

In [ ]:
n, t_call = 144, 3.6  # объектов в выборке, секунд на один вызов модели

saving_hours = ...  # ваш код: (время точного перебора - время аппроксимации) в часах
print(saving_hours)


# Корреляция признаков и перестановочная важность

Вернемся к игрушечному примеру из теории: $y = X_1 \text{ AND } X_2$ на данных, где $X_1=1$ всегда влечет $X_2=1$. Важность $X_2$ оцениваем через качество (accuracy) по двум перестановкам столбца $X_2$ из таблицы теории — $X_2^{*}$ и $X_2^{**}$:

$$PI(X_2) = q_{orig} - \frac{q_1 + q_2}{2}.$$

Модель A опирается только на $X_1$: $\hat y = X_1$. Модель B честно вычисляет оба признака: $\hat y = X_1 \text{ AND } X_2$.

**Q5. Игрушечный пример «побитовое И» ($y=X_1 \text{ AND } X_2$, $X_1{=}1\Rightarrow X_2{=}1$). Модель B честно вычисляет `X1 AND X2`. Чему равна перестановочная важность $PI_B(X_2)$ по двум перестановкам X<sub>2</sub>* и X<sub>2</sub>** из таблицы урока «Permutation importance»?**

Введите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/5) · [шаг на Степике](https://stepik.org/lesson/1329680/step/6)

In [ ]:
X1      = np.array([1, 1, 0, 0])
X2_orig = np.array([1, 1, 1, 0])
X2_star = np.array([0, 1, 1, 1])   # перестановка X2* из таблицы теории
X2_2star = np.array([1, 0, 1, 1])  # перестановка X2** из таблицы теории
y_toy = X1 & X2_orig

def model_A(x1, x2):
    return x1

def model_B(x1, x2):
    return x1 & x2

def accuracy(y_true, y_pred):
    return (y_true == y_pred).mean()

def toy_pi(model):
    q_orig = accuracy(y_toy, model(X1, X2_orig))
    q_1 = accuracy(y_toy, model(X1, X2_star))
    q_2 = accuracy(y_toy, model(X1, X2_2star))
    return q_orig - (q_1 + q_2) / 2

print("PI_A(X2) =", toy_pi(model_A))

PI_B_X2 = ...  # ваш код
print("PI_B(X2) =", PI_B_X2)


# Эффект Рашомона на California Housing

Обучим три почти одинаково точные линейные модели и сравним у них важность одного и того же признака. В этом задании и в следующих двух (Q7, Q8) перестановочную важность считаем функцией `permutation_importance` из sklearn с параметрами по умолчанию: для регрессии она сравнивает метрику $R^2$, так что $PI$ — это **падение $R^2$** на тестовых данных после перестановки признака. Число перестановок `n_repeats=10`, `random_state=42`. Все три модели стоят в конвейере со `StandardScaler`, у `ElasticNet` берем `alpha=0.01`.

**Q6\* (бонус). Три модели (`Linear`, `Ridge(alpha=50)`, `ElasticNet`) держат $R^2\in[0.591,\ 0.594]$ на тесте — один Rashomon-«коридор». Чему равен разброс (max − min) `PI(Longitude)` между ними? Ответ округлите до сотых.**

Введите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/6) · [шаг на Степике](https://stepik.org/lesson/1329680/step/7)

In [ ]:
rashomon_models = {
    'Linear': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge(alpha=50)': make_pipeline(StandardScaler(), Ridge(alpha=50)),
    'ElasticNet': make_pipeline(StandardScaler(), ElasticNet(alpha=0.01)),
}

pi_longitude = {}
for name, model in rashomon_models.items():
    model.fit(X_train, y_train)
    print(f"{name}: R^2 на тесте = {model.score(X_test, y_test):.4f}")
    pi_longitude[name] = ...  # ваш код: importances_mean признака Longitude из permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)

print(pi_longitude)
spread = ...  # ваш код: max - min
print("Разброс PI(Longitude):", spread)


# Условная перестановка

Маргинальная перестановка берет значения `Longitude` из всей тестовой выборки. Условная перемешивает `Longitude` только внутри групп объектов с похожей широтой: разобьем `Latitude` на 10 квантильных групп (`pd.qcut`) и переставим `Longitude` внутри каждой группы. Обе важности — падение $R^2$ модели `lin_model` на тестовых данных, по 10 перестановок с `seed=42`.

**Q7. Для `Longitude` (условие на `Latitude`) marginal PI ≈1.14, conditional PI ≈0.15. Выберите верное объяснение:**

Выберите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/7) · [шаг на Степике](https://stepik.org/lesson/1329680/step/8)

In [ ]:
def conditional_pi(model, X, y, feature, condition_on, n_bins=10, K=10, seed=42):
    rs = np.random.RandomState(seed)
    base = model.score(X, y)
    groups = pd.qcut(X[condition_on], n_bins, labels=False).values
    drops = []
    for k in range(K):
        values = X[feature].values.copy()
        for g in np.unique(groups):
            idx = np.where(groups == g)[0]
            values[idx] = values[rs.permutation(idx)]  # перестановка только внутри группы
        X_perm = X.copy()
        X_perm[feature] = values
        drops.append(base - model.score(X_perm, y))
    return np.mean(drops)

marginal = permutation_importance(lin_model, X_test, y_test, n_repeats=10, random_state=42)
marginal_pi = pd.Series(marginal.importances_mean, index=X_test.columns)

print("marginal PI(Longitude):", round(marginal_pi['Longitude'], 3))
print("conditional PI(Longitude | Latitude):", round(conditional_pi(lin_model, X_test, y_test, 'Longitude', 'Latitude'), 3))


# LOCO: выбросить признак и переобучить

LOCO (Leave-One-Covariate-Out) для признака — насколько выросла ошибка MSE на тестовых данных, если модель **переобучить** без этого признака: $LOCO_i = e_{-i} - e_{orig}$. Модель та же, что `lin_model`. Рядом — маргинальная перестановочная важность (падение $R^2$) из предыдущего задания.

**Q8. По результатам LOCO ранжирование сильно отличается от marginal PI. Выберите верные утверждения:**

Выберите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/8) · [шаг на Степике](https://stepik.org/lesson/1329680/step/9)

In [ ]:
e_full = mean_squared_error(y_test, lin_model.predict(X_test))
loco = {}
for feature in X_train.columns:
    reduced = make_pipeline(StandardScaler(), LinearRegression())
    reduced.fit(X_train.drop(columns=feature), y_train)  # переобучение без признака
    loco[feature] = mean_squared_error(y_test, reduced.predict(X_test.drop(columns=feature))) - e_full

comparison = pd.DataFrame({'marginal_PI': marginal_pi, 'LOCO': pd.Series(loco)})
comparison['rank_marginal'] = comparison['marginal_PI'].rank(ascending=False).astype(int)
comparison['rank_LOCO'] = comparison['LOCO'].rank(ascending=False).astype(int)
comparison.sort_values('LOCO', ascending=False).round(4)


# Corrected PIMP: отличима ли важность от шума

Corrected PIMP (Altmann и др., 2010) строит нулевое распределение важности: таргет обучающей выборки перемешивается $N=50$ раз, модель каждый раз переобучается, и для каждого признака заново считается `PI_ratio` (как в Q2–Q3: MSE на тестовых данных, $K=10$, `seed=42`). p-value признака — доля перемешиваний, где важность на перемешанном таргете не меньше настоящей: $p = \frac{\#\{PI^{null} \ge PI\} + 1}{N + 1}$.

**По результатам Corrected PIMP (50 перемешиваний таргета) какие признаки оказались неотличимы от шума (p-value > 0.25)?**

Выберите ответ в тренажере.

📍 Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/2/9) · [шаг на Степике](https://stepik.org/lesson/1329680/step/10)

In [ ]:
N_SHUFFLES = 50
observed = pi_lin['PI_ratio']

rs_target = np.random.RandomState(42)
null_ratios = []
for s in range(N_SHUFFLES):
    y_shuffled = rs_target.permutation(y_train.values)  # перемешанный таргет
    null_model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_shuffled)
    null_ratios.append(pi_table(null_model, X_test, y_test)['PI_ratio'])
null_ratios = pd.DataFrame(null_ratios)

p_values = ((null_ratios >= observed).sum() + 1) / (N_SHUFFLES + 1)
pimp = pd.DataFrame({'PI_ratio': observed, 'p_value': p_values})
print(pimp.sort_values('p_value').round(3))

noise_like = ...  # ваш код: признаки с p-value > 0.25
print(noise_like)
